# 🔬 ISOM 260: Build a Research Agent with ReAct Reasoning

**Session 8 — AI Agents & Automation** | Suffolk University | Prof. Hasan Arslan

---

## Recap: What You Built in Session 4

In Session 4 you learned the **Agent Equation**:

```
AI Agent = LLM + Tools + Reasoning Loop + Memory
```

You gave Claude a calculator and stock lookup tools. Claude **decided on its own** when to use each tool. That was powerful — but the agent just *reacted*. It didn't *reason* about its plan.

## What's New Today: ReAct Reasoning

**ReAct** = **Re**ason + **Act**

Instead of blindly calling tools, a ReAct agent:

1. 💭 **Thinks** — "What do I need to find out? What's the best tool for this?"
2. 🛠️ **Acts** — Calls a tool with specific inputs
3. 👁️ **Observes** — Reads the result and evaluates quality
4. 🔁 **Repeats** — Decides if it needs more info or is ready to synthesize

This is the exact pattern behind **Claude Code**, **ChatGPT Deep Research**, **Perplexity**, and every serious AI agent in production today.

Today you'll build a **Research Agent** that searches, evaluates sources, extracts facts, and writes structured reports — all with visible reasoning at every step.

---

## 🚀 Part 1: Setup (2 minutes)

Same setup as Session 4 — install the SDK and connect your API key.

In [ ]:
# Install the Anthropic Python SDK
!pip install -q anthropic

In [ ]:
# Set your API key
# Same key from Session 4 — it still works!
import os
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the key icon in the left sidebar → Add secret named ANTHROPIC_API_KEY
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets!")
except:
    # Option 2: Paste directly (less secure, but works for class)
    os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"  # <-- Replace this
    print("⚠️ Using hardcoded API key. Consider using Colab Secrets instead.")

In [ ]:
# Verify connection
import anthropic
import json

client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say 'Research Agent ready!' in exactly 3 words."}]
)

print(response.content[0].text)
print(f"\n✅ Connection successful! Using model: claude-sonnet-4-5-20250929")

---

## 🤔 Part 2: Naive vs. ReAct — Why Reasoning Matters

Before we build the full research agent, let's see the problem with a **naive** approach.

A naive agent just sends your question to Claude and returns whatever comes back. No tools, no reasoning trace, no source evaluation. It's basically a chatbot wearing an agent costume.

```
┌─────────────────────────────────────────────┐
│  NAIVE AGENT                                │
│  User: "Research AI's impact on jobs"        │
│  → Just asks Claude with no tools             │
│  → Gets a generic essay                       │
│  ❌ No sources. No evaluation. No structure.  │
└─────────────────────────────────────────────┘

┌─────────────────────────────────────────────┐
│  ReAct AGENT (what we're building)           │
│  User: "Research AI's impact on jobs"        │
│  THINK: Need data on job displacement + new  │
│        roles. Search authoritative sources.   │
│  → web_search("AI impact on job market")      │
│  OBSERVE: Found McKinsey, WEF, Reuters data  │
│  THINK: McKinsey is credible. Let me verify.  │
│  → evaluate_source("McKinsey")                │
│  OBSERVE: 9/10 credibility. Slight biz bias.  │
│  THINK: Good. Now extract key numbers.        │
│  → extract_key_facts(results)                 │
│  ✅ Reasoned. Evaluated. Synthesized.         │
└─────────────────────────────────────────────┘
```

Let's prove it.

In [ ]:
# ============================================
# The NAIVE approach: just ask Claude directly
# ============================================
# No tools. No reasoning. No source evaluation.
# This is what most people do — and it's not an agent.

def naive_agent(query: str) -> str:
    """
    A 'naive agent' that just passes the query to Claude.
    No tools, no reasoning trace, no source evaluation.
    """
    response = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=1024,
        messages=[{"role": "user", "content": query}]
    )
    return response.content[0].text

print("✅ Naive agent defined. Let's see what it produces...")

In [ ]:
# Test the naive agent
print("\n" + "="*60)
print("💤 NAIVE AGENT: No tools, no reasoning")
print("="*60)

naive_result = naive_agent("Research the impact of AI on the job market")
print(naive_result)

print("\n" + "-"*60)
print("⚠️ Notice: No sources cited. No credibility check.")
print("   No reasoning trace. Just a generic essay.")
print("   How do you know any of this is accurate?")
print("-"*60)

### The Problem

That output *looks* good but:
- **No sources** — Where did those numbers come from? Are they real?
- **No evaluation** — How credible is the information?
- **No reasoning** — Why did it focus on those topics?
- **No structure** — Just a wall of text, not an actionable briefing

Now let's build the ReAct version that fixes ALL of these problems. First, we need tools.

---

## 🛠️ Part 3: Build 3 Research Tools

Just like Session 4, we need to give Claude tools. But this time our tools are designed for **research**:

| Tool | Purpose | Real-World Equivalent |
|------|---------|----------------------|
| `web_search` | Find information on a topic | Google, Perplexity |
| `evaluate_source` | Rate source credibility | Media Bias/Fact Check |
| `extract_key_facts` | Pull structured data from text | Research assistant |

Each tool follows the same pattern from Session 4:
1. **Tool definition** — tells Claude what it can do (the "job description")
2. **Python function** — the actual code that runs
3. **Test** — make sure it works before giving it to the agent

In [ ]:
# ============================================
# TOOL 1: web_search
# Simulated web search with realistic results
# ============================================
# In production, this would hit a real search API (Google, Bing, Tavily).
# For class, we use pre-loaded results so it works without API keys.

# Tool definition — this tells Claude what the tool can do
web_search_tool = {
    "name": "web_search",
    "description": (
        "Search the web for information on a topic. Returns a list of search results "
        "with titles, sources, snippets, and credibility scores. Use this tool to find "
        "data, statistics, expert opinions, and news articles. You can search for any "
        "topic — the more specific your query, the better the results."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query. Be specific for better results, e.g. 'AI job displacement statistics 2025' instead of just 'AI jobs'"
            }
        },
        "required": ["query"]
    }
}

# Python function — the actual search implementation
def web_search(query: str) -> str:
    """
    Simulated web search with pre-loaded results for common research topics.
    Returns JSON array of search results.
    """
    # Pre-loaded search results for various topics
    search_database = {
        "ai job market": [
            {
                "title": "AI Will Transform 40% of Jobs Worldwide, IMF Says",
                "source": "Reuters",
                "snippet": "The IMF estimates AI will affect nearly 40% of global employment. In advanced economies, about 60% of jobs are exposed to AI, with roughly half benefiting from AI integration and the other half facing displacement risk.",
                "credibility_score": 9,
                "date": "2025-01"
            },
            {
                "title": "McKinsey: AI Could Automate 30% of Work Hours by 2030",
                "source": "McKinsey Global Institute",
                "snippet": "Generative AI could automate activities that absorb 60-70% of employees' time today. By 2030, up to 30% of hours currently worked could be automated, with midpoint estimates suggesting 15% of current work activities could be displaced.",
                "credibility_score": 9,
                "date": "2024-06"
            },
            {
                "title": "World Economic Forum: 85 Million Jobs Displaced, 97 Million Created",
                "source": "World Economic Forum",
                "snippet": "The Future of Jobs Report projects that AI and automation will displace 85 million jobs by 2025 but create 97 million new roles. Net positive of 12 million jobs, but massive reskilling needed. Top growing roles: AI specialists, data analysts, digital transformation experts.",
                "credibility_score": 9,
                "date": "2024-09"
            },
            {
                "title": "AI Job Market: Winners and Losers in the New Economy",
                "source": "Harvard Business Review",
                "snippet": "Jobs requiring routine cognitive tasks face the highest automation risk (data entry, basic analysis, customer service). Creative, strategic, and interpersonal roles are growing. The key skill shift: from 'doing tasks' to 'managing AI that does tasks.'",
                "credibility_score": 8,
                "date": "2025-02"
            },
            {
                "title": "AI Jobs Boom: Tech Salaries Hit Record as Companies Scramble for Talent",
                "source": "TechCrunch Blog",
                "snippet": "Average AI engineer salary now exceeds $200K. Companies are paying 40% premiums for AI skills. However, critics say the boom is concentrated in a few cities and roles, leaving most workers behind.",
                "credibility_score": 6,
                "date": "2025-03"
            }
        ],
        "ev market": [
            {
                "title": "Global EV Sales Surpass 17 Million in 2024, Up 25% YoY",
                "source": "International Energy Agency (IEA)",
                "snippet": "Global electric vehicle sales reached 17 million units in 2024, a 25% increase from 2023. China accounted for 60% of global EV sales. Battery costs fell 14% to $115/kWh. The IEA projects EVs will represent 50% of new car sales by 2030.",
                "credibility_score": 10,
                "date": "2025-01"
            },
            {
                "title": "Tesla's Market Share Slips as Competition Intensifies",
                "source": "Bloomberg",
                "snippet": "Tesla's global EV market share dropped from 19% to 14% in 2024 as Chinese competitors BYD, NIO, and Xpeng gained ground. BYD surpassed Tesla in total vehicle sales for Q4 2024. Legacy automakers VW, BMW, and Hyundai also gaining rapidly.",
                "credibility_score": 9,
                "date": "2025-02"
            },
            {
                "title": "EV Infrastructure: 500K Public Chargers Now in the US",
                "source": "Department of Energy",
                "snippet": "The US crossed 500,000 public EV charging points in late 2024. Government target: 1.2 million by 2030. Range anxiety declining but rural coverage remains sparse. Average charging time dropped 30% with new fast-charging standards.",
                "credibility_score": 10,
                "date": "2024-12"
            },
            {
                "title": "Are EVs Actually Cheaper? Total Cost of Ownership Analysis",
                "source": "Consumer Reports",
                "snippet": "Over a 10-year period, EVs cost $6,000-$12,000 less than comparable gas vehicles when factoring in fuel, maintenance, and insurance. Upfront costs remain 15-20% higher but the gap is closing. Used EV prices dropped 25% in 2024.",
                "credibility_score": 8,
                "date": "2025-01"
            }
        ],
        "remote work trends": [
            {
                "title": "Remote Work Stabilizes: 28% of Work Days Now Remote",
                "source": "Stanford University / WFH Research",
                "snippet": "After post-pandemic adjustments, remote work has stabilized at 28% of all work days in the US. Hybrid (3 days in-office) is the dominant model for knowledge workers. Fully remote roles have decreased from peak of 33% to about 15% of job postings.",
                "credibility_score": 9,
                "date": "2025-02"
            },
            {
                "title": "Remote Workers Are 13% More Productive, Meta-Analysis Finds",
                "source": "Harvard Business Review",
                "snippet": "A comprehensive meta-analysis of 42 studies finds remote workers are 13% more productive on average, with the highest gains in focused individual work. Collaboration and mentoring suffer. Hybrid appears to capture most benefits while mitigating downsides.",
                "credibility_score": 8,
                "date": "2024-11"
            },
            {
                "title": "Return-to-Office Mandates Backfire: Top Talent Leaving",
                "source": "Wall Street Journal",
                "snippet": "Companies with strict RTO mandates are losing senior employees at 2x the normal rate. Amazon, Dell, and JPMorgan faced significant pushback. Meanwhile, companies offering flexibility report 25% higher applicant rates for open roles.",
                "credibility_score": 8,
                "date": "2025-01"
            }
        ],
        "cryptocurrency regulation": [
            {
                "title": "EU's MiCA Regulation Now Fully in Effect for Crypto Markets",
                "source": "Reuters",
                "snippet": "The EU's Markets in Crypto-Assets (MiCA) regulation is now fully enforced, requiring all crypto service providers to obtain licenses. Early data shows 30% of smaller exchanges exiting the EU market while larger players like Coinbase and Kraken gained market share.",
                "credibility_score": 9,
                "date": "2025-01"
            },
            {
                "title": "US SEC Approves Spot Bitcoin ETFs, $50B Flows In",
                "source": "Bloomberg",
                "snippet": "Since SEC approval of spot Bitcoin ETFs in January 2024, over $50 billion has flowed into these products. BlackRock's iShares Bitcoin Trust leads with $20B AUM. The approval is credited with Bitcoin's rise from $42K to over $95K.",
                "credibility_score": 9,
                "date": "2025-02"
            },
            {
                "title": "Crypto Scams Cost Consumers $5.6 Billion in 2024",
                "source": "FBI Internet Crime Report",
                "snippet": "The FBI reports crypto-related fraud losses of $5.6 billion in 2024, up 45% from 2023. Investment scams (pig butchering) accounted for 71% of losses. Regulators cite consumer protection as primary motivation for tighter rules.",
                "credibility_score": 10,
                "date": "2025-03"
            }
        ],
        "climate tech investment": [
            {
                "title": "Climate Tech VC Investment Hits $65B in 2024",
                "source": "PitchBook",
                "snippet": "Venture capital investment in climate technology reached $65 billion in 2024, up 18% from 2023. Solar and battery storage led deal volume. Carbon capture attracted the most new investors, though returns remain unproven. Top deal: $2.1B for a solid-state battery startup.",
                "credibility_score": 9,
                "date": "2025-01"
            },
            {
                "title": "Green Hydrogen Costs Fall 40%, Reaching Grid Parity in Key Markets",
                "source": "International Renewable Energy Agency (IRENA)",
                "snippet": "Green hydrogen production costs fell 40% between 2022 and 2024, reaching $3-4/kg in optimal locations. At $2/kg (projected by 2028), it becomes competitive with natural gas for industrial heating. Major projects underway in Saudi Arabia, Australia, and Chile.",
                "credibility_score": 10,
                "date": "2024-12"
            },
            {
                "title": "AI's Carbon Footprint: Data Centers Now 4% of Global Electricity",
                "source": "Nature",
                "snippet": "Data centers consumed approximately 4% of global electricity in 2024, up from 2% in 2022, driven largely by AI training and inference. The IEA projects this could reach 6% by 2028. Tech companies racing to secure renewable energy contracts.",
                "credibility_score": 10,
                "date": "2025-02"
            }
        ],
        "healthcare ai": [
            {
                "title": "AI Diagnostics Match or Exceed Doctors in 14 Specialties",
                "source": "The Lancet Digital Health",
                "snippet": "A comprehensive review of 82 studies shows AI diagnostic systems match or exceed physician accuracy in 14 medical specialties, including radiology, pathology, and dermatology. However, AI performs worse when patient demographics differ from training data.",
                "credibility_score": 10,
                "date": "2025-01"
            },
            {
                "title": "FDA Clears 700+ AI Medical Devices, Regulation Struggles to Keep Pace",
                "source": "STAT News",
                "snippet": "The FDA has now cleared over 700 AI-enabled medical devices. Radiology accounts for 75% of clearances. Critics argue the 510(k) pathway is too lenient for AI, as models can drift after deployment. Post-market surveillance remains inadequate.",
                "credibility_score": 8,
                "date": "2025-02"
            },
            {
                "title": "Hospital System Saves $12M Annually Using AI for Admin Tasks",
                "source": "Healthcare IT News",
                "snippet": "A major hospital network reported $12M in annual savings by deploying AI for clinical documentation, scheduling, and insurance pre-authorization. Physician burnout scores improved 22%. Key success factor: AI handles paperwork, humans handle patients.",
                "credibility_score": 7,
                "date": "2024-11"
            }
        ]
    }

    # Find the best matching topic
    query_lower = query.lower()
    best_match = None
    best_score = 0

    for topic, results in search_database.items():
        # Count how many words from the topic appear in the query
        topic_words = topic.split()
        score = sum(1 for word in topic_words if word in query_lower)
        if score > best_score:
            best_score = score
            best_match = topic

    if best_match and best_score > 0:
        results = search_database[best_match]
        return json.dumps({
            "query": query,
            "results_count": len(results),
            "results": results
        }, indent=2)
    else:
        return json.dumps({
            "query": query,
            "results_count": 0,
            "results": [],
            "note": f"No results found for '{query}'. Try searching for: AI job market, EV market, remote work trends, cryptocurrency regulation, climate tech investment, or healthcare AI."
        }, indent=2)

# Test it!
print("🔍 Testing web_search tool:")
print("-" * 40)
result = json.loads(web_search("AI impact on job market"))
print(f"Query: 'AI impact on job market'")
print(f"Results found: {result['results_count']}")
for r in result['results'][:2]:  # Show first 2
    print(f"  📰 {r['title']} ({r['source']}) - Credibility: {r['credibility_score']}/10")
print("\n✅ web_search tool is working!")

In [ ]:
# ============================================
# TOOL 2: evaluate_source
# Checks the credibility of a source
# ============================================
# In production, this would cross-reference media bias databases.
# For class, we use a curated list of well-known sources.

# Tool definition
evaluate_source_tool = {
    "name": "evaluate_source",
    "description": (
        "Evaluate the credibility of a news source or publication. Returns a credibility "
        "score (1-10), bias notes, and a recommendation on how much to trust information "
        "from this source. Use this to verify the reliability of search results before "
        "including them in your analysis."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "source_name": {
                "type": "string",
                "description": "The name of the source to evaluate, e.g. 'Reuters', 'TechCrunch', 'Wikipedia'"
            }
        },
        "required": ["source_name"]
    }
}

# Python function
def evaluate_source(source_name: str) -> str:
    """
    Evaluate the credibility of a given source.
    Returns JSON with credibility_score, bias_notes, and recommendation.
    """
    source_ratings = {
        "reuters": {
            "credibility_score": 9,
            "type": "Wire Service",
            "bias_notes": "Minimal bias. Fact-based reporting. Considered one of the most reliable global news sources.",
            "recommendation": "Highly trustworthy. Use as primary source."
        },
        "bloomberg": {
            "credibility_score": 9,
            "type": "Financial News",
            "bias_notes": "Slight pro-business lean. Excellent financial data accuracy. Strong editorial standards.",
            "recommendation": "Highly trustworthy for financial and business data. Cross-check opinion pieces."
        },
        "mckinsey global institute": {
            "credibility_score": 8,
            "type": "Consulting Research",
            "bias_notes": "Pro-business and pro-technology perspective. Research is thorough but tends to emphasize positive transformation narratives. May understate disruption risks.",
            "recommendation": "Good for data and trends. Balance with labor/policy perspectives."
        },
        "world economic forum": {
            "credibility_score": 8,
            "type": "International Organization",
            "bias_notes": "Pro-globalization, pro-technology, pro-business perspective. Research is well-funded but reflects stakeholder interests.",
            "recommendation": "Reliable for macro trends and data. Be aware of institutional perspective."
        },
        "harvard business review": {
            "credibility_score": 8,
            "type": "Academic/Business Publication",
            "bias_notes": "Pro-management perspective. Peer-reviewed for academic rigor. Articles range from research-backed to opinion.",
            "recommendation": "Trustworthy for management insights. Distinguish research articles from opinion."
        },
        "techcrunch blog": {
            "credibility_score": 6,
            "type": "Tech Blog",
            "bias_notes": "Strong pro-startup, pro-VC bias. Often hype-driven. Good for early trend spotting but may overstate market potential.",
            "recommendation": "Use for tech industry pulse. Cross-check claims with more rigorous sources."
        },
        "international energy agency (iea)": {
            "credibility_score": 10,
            "type": "International Organization",
            "bias_notes": "Considered the gold standard for energy data. Methodologies are transparent and peer-reviewed. Very conservative projections.",
            "recommendation": "Highest credibility. Use as primary source for energy data."
        },
        "department of energy": {
            "credibility_score": 9,
            "type": "Government Agency",
            "bias_notes": "Official US government data. May reflect policy priorities of current administration. Data itself is reliable.",
            "recommendation": "Highly trustworthy for US-specific data. Note potential policy framing."
        },
        "consumer reports": {
            "credibility_score": 8,
            "type": "Consumer Advocacy",
            "bias_notes": "Pro-consumer perspective. Independent testing methodology. No advertising revenue ensures objectivity.",
            "recommendation": "Very reliable for product comparisons and consumer data."
        },
        "wall street journal": {
            "credibility_score": 8,
            "type": "Major Newspaper",
            "bias_notes": "Center-right editorial lean, especially on opinion pages. News reporting is factual and well-sourced.",
            "recommendation": "Trustworthy for news reporting. Distinguish news from opinion pages."
        },
        "stanford university / wfh research": {
            "credibility_score": 9,
            "type": "Academic Research",
            "bias_notes": "Led by Prof. Nick Bloom, the leading researcher on remote work. Peer-reviewed methodology. Large sample sizes.",
            "recommendation": "Gold standard for remote work data. Highly credible."
        },
        "the lancet digital health": {
            "credibility_score": 10,
            "type": "Peer-Reviewed Medical Journal",
            "bias_notes": "Rigorous peer review. Part of The Lancet family, one of the most respected medical publications. Very high standards.",
            "recommendation": "Highest credibility for medical/health technology research."
        },
        "nature": {
            "credibility_score": 10,
            "type": "Peer-Reviewed Scientific Journal",
            "bias_notes": "Premier scientific journal. Rigorous peer review. Highest standard of evidence.",
            "recommendation": "Highest credibility for scientific data and analysis."
        },
        "pitchbook": {
            "credibility_score": 9,
            "type": "Financial Data Provider",
            "bias_notes": "Industry-standard VC/PE data. Comprehensive deal tracking. Pro-investment perspective but data is reliable.",
            "recommendation": "Highly trustworthy for investment and startup data."
        },
        "fbi internet crime report": {
            "credibility_score": 10,
            "type": "Government Report",
            "bias_notes": "Official law enforcement data. Based on actual reported crimes. May undercount (not all crimes are reported).",
            "recommendation": "Highly authoritative. Note that actual figures may be higher than reported."
        },
        "stat news": {
            "credibility_score": 8,
            "type": "Health/Science News",
            "bias_notes": "Specialized health and life sciences reporting. Well-sourced journalism. Occasionally sensationalizes headlines.",
            "recommendation": "Very good for healthcare industry coverage. Read past headlines for nuance."
        },
        "wikipedia": {
            "credibility_score": 7,
            "type": "Crowdsourced Encyclopedia",
            "bias_notes": "Quality varies by article. Well-cited articles are reliable for background. Can be edited by anyone. Best for established facts, not current events.",
            "recommendation": "Good starting point. Always check the cited sources directly."
        },
        "random blog": {
            "credibility_score": 3,
            "type": "Personal Blog",
            "bias_notes": "No editorial oversight. No fact-checking. Author credentials unknown. May contain opinions presented as facts.",
            "recommendation": "Do not use as primary source. Cross-check all claims with credible sources."
        }
    }

    source_lower = source_name.lower().strip()

    # Try exact match first, then partial match
    if source_lower in source_ratings:
        result = source_ratings[source_lower]
        result["source_name"] = source_name
        return json.dumps(result, indent=2)

    # Partial match
    for key, rating in source_ratings.items():
        if source_lower in key or key in source_lower:
            result = rating.copy()
            result["source_name"] = source_name
            return json.dumps(result, indent=2)

    # Unknown source
    return json.dumps({
        "source_name": source_name,
        "credibility_score": 5,
        "type": "Unknown",
        "bias_notes": "This source is not in our credibility database. Unable to assess bias or reliability.",
        "recommendation": "Treat with caution. Cross-reference claims with well-known, established sources."
    }, indent=2)

# Test it!
print("📊 Testing evaluate_source tool:")
print("-" * 40)
for source in ["Reuters", "TechCrunch Blog", "Random Blog"]:
    result = json.loads(evaluate_source(source))
    print(f"  {source}: {result['credibility_score']}/10 — {result['recommendation']}")
print("\n✅ evaluate_source tool is working!")

In [ ]:
# ============================================
# TOOL 3: extract_key_facts
# Pulls structured facts from text
# ============================================
# In production, this might use NLP or another LLM.
# For class, we use simple parsing to find numbers and claims.

import re

# Tool definition
extract_key_facts_tool = {
    "name": "extract_key_facts",
    "description": (
        "Extract structured key facts from a block of text. Returns an array of facts "
        "including statistics (numbers, percentages, dollar amounts), key claims, and "
        "named entities (organizations, people, locations). Use this after searching to "
        "distill raw search results into structured data points for your analysis."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "text": {
                "type": "string",
                "description": "The text to extract facts from. Can be a search result snippet, article excerpt, or any text block."
            }
        },
        "required": ["text"]
    }
}

# Python function
def extract_key_facts(text: str) -> str:
    """
    Extract structured key facts from text.
    Finds numbers, percentages, dollar amounts, and key entities.
    Returns JSON with categorized facts.
    """
    facts = {
        "statistics": [],
        "claims": [],
        "entities": []
    }

    # Extract statistics: numbers with context
    # Percentages
    pct_matches = re.findall(r'[\d,.]+%', text)
    for match in pct_matches:
        # Find surrounding context
        idx = text.find(match)
        start = max(0, idx - 60)
        end = min(len(text), idx + len(match) + 40)
        context = text[start:end].strip()
        if context.startswith(',') or context.startswith(' '):
            context = context.lstrip(', ')
        facts["statistics"].append({
            "value": match,
            "type": "percentage",
            "context": context
        })

    # Dollar amounts
    dollar_matches = re.findall(r'\$[\d,.]+[BMKTbmkt]?(?:\s*(?:billion|million|trillion))?', text)
    for match in dollar_matches:
        idx = text.find(match)
        start = max(0, idx - 50)
        end = min(len(text), idx + len(match) + 40)
        context = text[start:end].strip()
        facts["statistics"].append({
            "value": match,
            "type": "monetary",
            "context": context
        })

    # Large numbers (millions, billions)
    big_num_matches = re.findall(r'[\d,.]+\s*(?:million|billion|trillion)', text, re.IGNORECASE)
    for match in big_num_matches:
        idx = text.find(match)
        start = max(0, idx - 50)
        end = min(len(text), idx + len(match) + 40)
        context = text[start:end].strip()
        facts["statistics"].append({
            "value": match.strip(),
            "type": "large_number",
            "context": context
        })

    # Extract key claims: sentences with strong verbs or indicators
    claim_indicators = [
        'found', 'shows', 'reveals', 'indicates', 'suggests',
        'projects', 'estimates', 'reports', 'predicts', 'concludes',
        'surpass', 'exceed', 'dropped', 'increased', 'decreased', 'grew'
    ]
    sentences = re.split(r'[.!]\s', text)
    for sentence in sentences:
        sentence = sentence.strip()
        if any(indicator in sentence.lower() for indicator in claim_indicators):
            if len(sentence) > 20:  # Skip very short fragments
                facts["claims"].append(sentence.rstrip('.'))

    # Extract named entities: capitalized multi-word names
    entity_matches = re.findall(r'[A-Z][a-z]+(?:\s[A-Z][a-z]+)+', text)
    # Also look for known organizations and acronyms
    org_patterns = re.findall(r'\b(?:IMF|IEA|FDA|FBI|SEC|EU|WEF|WHO|UN|NATO|IRENA)\b', text)
    entities = list(set(entity_matches + org_patterns))
    facts["entities"] = entities

    # Summary
    facts["summary"] = {
        "total_statistics": len(facts["statistics"]),
        "total_claims": len(facts["claims"]),
        "total_entities": len(facts["entities"])
    }

    return json.dumps(facts, indent=2)

# Test it!
print("📝 Testing extract_key_facts tool:")
print("-" * 40)
test_text = (
    "The IMF estimates AI will affect nearly 40% of global employment. "
    "In advanced economies, about 60% of jobs are exposed to AI. "
    "McKinsey projects that $4.4 trillion in annual productivity gains are possible."
)
result = json.loads(extract_key_facts(test_text))
print(f"Statistics found: {result['summary']['total_statistics']}")
print(f"Claims found: {result['summary']['total_claims']}")
print(f"Entities found: {result['summary']['total_entities']}")
for stat in result['statistics'][:3]:
    print(f"  📊 {stat['type']}: {stat['value']}")
print("\n✅ extract_key_facts tool is working!")

### 🌟 Checkpoint: 3 Tools Ready

You now have a research toolkit:

| Tool | What It Does | When the Agent Uses It |
|------|-------------|------------------------|
| `web_search` | Finds information | Agent needs data on a topic |
| `evaluate_source` | Rates credibility | Agent wants to verify a source is trustworthy |
| `extract_key_facts` | Pulls out key numbers and claims | Agent wants to structure raw text into data |

Now let's give these tools to Claude with **ReAct reasoning** so it knows *how to think* while using them.

---

## 🧠 Part 4: Wire the ReAct Loop

This is the upgrade from Session 4. Same agent loop concept, but now Claude **thinks out loud** at every step:

```
┌───────────────────────────────────────┐
│ ReAct Loop                              │
│                                         │
│  1. THOUGHT: "I need to find X..."      │
│     ↓                                    │
│  2. ACTION: web_search("X")             │
│     ↓                                    │
│  3. OBSERVATION: [results]               │
│     ↓                                    │
│  4. THOUGHT: "Interesting. Source Y       │
│     looks credible. Let me verify..."     │
│     ↓                                    │
│  5. ACTION: evaluate_source("Y")         │
│     ↓                                    │
│  6. OBSERVATION: [credibility data]       │
│     ↓                                    │
│  7. THOUGHT: "I have enough info.         │
│     Let me synthesize."                   │
│     ↓                                    │
│  8. FINAL ANSWER                         │
└───────────────────────────────────────┘
```

The key upgrade: Claude's **system prompt** tells it to reason explicitly before every action.

In [ ]:
# ============================================
# THE ReAct AGENT LOOP
# ============================================
# This is the same core pattern as Session 4, but upgraded with:
#   1. A ReAct-style system prompt that forces explicit reasoning
#   2. Verbose step-by-step trace printing
#   3. Formatted Thought / Action / Observation labels

def run_react_agent(user_message: str, tools: list, tool_functions: dict, max_steps: int = 10):
    """
    Run a ReAct research agent.

    ReAct = Reason + Act. The agent thinks before every action,
    observes the result, and reflects before deciding the next step.

    This is the exact pattern used by Claude Code, Perplexity,
    and every production AI agent.
    """
    # The ReAct system prompt — this is what makes the agent THINK
    system_prompt = (
        "You are a research agent that uses the ReAct (Reason + Act) framework. "
        "For every task, follow this pattern:\n\n"
        "1. THINK: Before each action, explain your reasoning. What do you know? "
        "What do you need to find out? Which tool is best for this?\n"
        "2. ACT: Use a tool to gather information.\n"
        "3. OBSERVE: After receiving results, reflect on what you learned. "
        "Is this source credible? Do you need more information?\n"
        "4. REPEAT or CONCLUDE: Either search for more information or synthesize your findings.\n\n"
        "Always think step by step. Evaluate source credibility before trusting data. "
        "When you have enough high-quality information, provide a clear, structured answer "
        "with specific data points and source attributions.\n\n"
        "Be thorough but efficient. Aim for 3-5 research steps before synthesizing."
    )

    print(f"\n{'='*60}")
    print(f"💬 User: {user_message}")
    print(f"{'='*60}")

    messages = [{"role": "user", "content": user_message}]
    step = 0

    while step < max_steps:
        step += 1

        # Send message to Claude with tools and ReAct system prompt
        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=4096,
            system=system_prompt,
            tools=tools,
            messages=messages
        )

        print(f"\n{'- '*30}")
        print(f"🔄 Step {step} | Stop reason: {response.stop_reason}")

        # Check if Claude wants to use a tool
        if response.stop_reason == "tool_use":
            tool_results = []

            for block in response.content:
                if block.type == "text" and block.text.strip():
                    # This is the THOUGHT step — Claude's reasoning
                    print(f"\n   💭 THOUGHT:")
                    # Print thought with indentation for readability
                    for line in block.text.strip().split('\n'):
                        print(f"      {line}")

                elif block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input

                    # This is the ACTION step
                    print(f"\n   🎯 ACTION: {tool_name}")
                    print(f"      Input: {json.dumps(tool_input, indent=2)[:200]}")

                    # Execute the tool
                    if tool_name in tool_functions:
                        result = tool_functions[tool_name](**tool_input)
                    else:
                        result = json.dumps({"error": f"Unknown tool: {tool_name}"})

                    # This is the OBSERVATION step
                    print(f"\n   👁️  OBSERVATION:")
                    # Print a truncated version of the result for readability
                    result_preview = result[:300] + "..." if len(result) > 300 else result
                    for line in result_preview.split('\n')[:8]:
                        print(f"      {line}")
                    if len(result) > 300:
                        print(f"      [...truncated for display, full data sent to agent]")

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            # Add to conversation history
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})

        else:
            # Claude is done — this is the FINAL ANSWER
            final_answer = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_answer += block.text

            print(f"\n{'='*60}")
            print(f"🤖 FINAL ANSWER:")
            print(f"{'='*60}")
            print(final_answer)
            print(f"\n✅ Research complete in {step} step(s)")

            return {"answer": final_answer, "messages": messages, "steps": step}

    # Safety: max steps reached
    print(f"\n⚠️ Max steps ({max_steps}) reached. Returning partial results.")
    return {"answer": "Max steps reached.", "messages": messages, "steps": step}

print("✅ ReAct agent loop defined!")
print("   Key upgrade from Session 4: Claude now THINKS before every ACTION.")

---

## 🔬 Part 5: Full Research Agent in Action

Time to wire everything together and watch the agent research real topics. Pay attention to the **reasoning trace** — you can see exactly *why* the agent makes each decision.

In [ ]:
# ============================================
# Register all 3 research tools
# ============================================

research_tools = [web_search_tool, evaluate_source_tool, extract_key_facts_tool]

research_functions = {
    "web_search": web_search,
    "evaluate_source": evaluate_source,
    "extract_key_facts": extract_key_facts
}

print("✅ Research toolkit registered:")
for t in research_tools:
    print(f"   🛠️  {t['name']} — {t['description'][:60]}...")

In [ ]:
# ============================================
# TEST 1: AI and the Job Market
# ============================================
# Watch the agent: search, evaluate sources, extract facts, synthesize

result_1 = run_react_agent(
    "Research the impact of AI on the job market. "
    "I need specific data, credible sources, and a balanced view.",
    research_tools,
    research_functions
)

In [ ]:
# ============================================
# TEST 2: EV Market Analysis
# ============================================
# Different topic, same reasoning pattern

result_2 = run_react_agent(
    "What's happening in the EV market? Give me the key trends, "
    "major players, and investment outlook.",
    research_tools,
    research_functions
)

### 🌟 What Just Happened

Compare that to the naive agent from Part 2. The ReAct agent:

1. **Searched** for relevant data (not just hallucinating statistics)
2. **Evaluated** source credibility (not treating all sources equally)
3. **Extracted** specific facts and numbers (not vague claims)
4. **Synthesized** a structured answer with source attribution
5. **Showed its reasoning** at every step (you can audit the process)

This is exactly what makes tools like **ChatGPT Deep Research** and **Perplexity** so much better than basic chatbots — they don't just generate text, they **research then generate**.

Now let's add one more power move: a structured research report.

In [ ]:
# ============================================
# BONUS: Generate a Structured Research Report
# ============================================
# Takes the agent's full conversation and asks Claude to
# write a professional briefing document.

def generate_report(agent_result: dict, topic: str) -> str:
    """
    Takes the full agent conversation and generates a structured
    research briefing with executive summary, findings, and recommendations.
    """
    # Build a summary of the conversation for the report
    conversation_text = ""
    for msg in agent_result["messages"]:
        if msg["role"] == "assistant" and isinstance(msg["content"], list):
            for block in msg["content"]:
                if hasattr(block, "text") and block.text:
                    conversation_text += block.text + "\n"
        elif msg["role"] == "assistant" and isinstance(msg["content"], str):
            conversation_text += msg["content"] + "\n"

    # Add the final answer
    conversation_text += "\nFINAL AGENT RESPONSE:\n" + agent_result["answer"]

    report_prompt = f"""Based on the following research agent conversation about "{topic}",
write a structured research briefing. The agent searched multiple sources,
evaluated their credibility, and extracted key facts.

AGENT CONVERSATION:
{conversation_text}

Write the briefing with this EXACT structure:

# Research Briefing: [Topic]

## Executive Summary
(2-3 sentences capturing the key finding)

## Key Findings
(Numbered list of 4-6 specific findings with data points)

## Source Quality Assessment
(Brief table or list of sources used and their credibility)

## Confidence Level
(High/Medium/Low with justification based on source quality and data consistency)

## Recommended Next Steps
(3-4 actionable recommendations for someone making business decisions on this topic)

Keep it concise and professional. Every claim should reference a source."""

    response = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=2048,
        messages=[{"role": "user", "content": report_prompt}]
    )

    return response.content[0].text

print("✅ Report generator defined!")

In [ ]:
# ============================================
# Generate a report from the AI job market research
# ============================================

print("📝 Generating structured research briefing...")
print("="*60)

report = generate_report(result_1, "Impact of AI on the Job Market")
print(report)

print("\n" + "="*60)
print("🌟 This is what a ReAct research agent produces:")
print("   → Structured, sourced, confidence-rated research.")
print("   → Not a generic essay — an actionable briefing.")

---

## 🎨 Part 6: Build Your Own Tool (Challenge)

Your research agent has 3 tools. But real research agents need more. Your challenge: **add a 4th tool** to make the agent more powerful.

### Idea Starters:

| Tool Idea | What It Does | Why It's Useful |
|-----------|-------------|------------------|
| `competitor_lookup` | Look up info about a specific company | Competitive analysis research |
| `financial_data` | Get revenue, growth, market cap for a company | Quantitative business research |
| `sentiment_analyzer` | Analyze if text is positive/negative/neutral | Gauge market sentiment |
| `trend_tracker` | Show how a topic has changed over time | Spot patterns and shifts |

### Template — Follow the same 3-step pattern as the tools above:

In [ ]:
# ============================================
# YOUR 4th TOOL — Fill in the blanks!
# ============================================

# Step 1: Tool definition (tells Claude what this tool can do)
my_research_tool = {
    "name": "your_tool_name",             # <-- Change this
    "description": (
        "Describe what your tool does. Be detailed! "  # <-- Change this
        "Tell Claude when to use it and what it returns."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "param1": {                      # <-- Change parameter name
                "type": "string",
                "description": "What this parameter means"  # <-- Change this
            }
        },
        "required": ["param1"]
    }
}

# Step 2: Python function (the actual implementation)
def your_tool_name(param1: str) -> str:    # <-- Match the tool name!
    """
    Your tool logic here.
    Use a dictionary of pre-loaded data (like the other tools).
    Must return a string (JSON-serialized is best).
    """
    # Your data dictionary here:
    data = {
        "example_key": {"info": "your data here"}
    }

    # Your logic here:
    result = data.get(param1.lower(), {"error": "Not found"})
    return json.dumps(result, indent=2)


# Step 3: Register all tools (including your new one)
my_tools = [web_search_tool, evaluate_source_tool, extract_key_facts_tool, my_research_tool]
my_functions = {
    "web_search": web_search,
    "evaluate_source": evaluate_source,
    "extract_key_facts": extract_key_facts,
    "your_tool_name": your_tool_name          # <-- Match the tool name!
}

# Step 4: Test it with a query that should trigger your new tool
# run_react_agent(
#     "Your test question here",              # <-- Change this
#     my_tools,
#     my_functions
# )

---

## 💭 Part 7: Reflection — The Bigger Picture

### What You Built vs. Production Multi-Agent Systems

| What You Built Today | Production Systems (ChatGPT Deep Research, Perplexity, Claude Code) |
|---|---|
| 3 simulated research tools | Real APIs: Google Search, databases, document stores |
| Single-agent ReAct loop | Multi-agent pipelines: Researcher → Writer → Editor |
| Reasoning printed to console | Reasoning stored, auditable, used for improvement |
| Pre-loaded search data | Live web search with billions of pages |
| Manual report generation | Automatic report formatting, citations, exports |
| ~80 lines of agent code | Thousands of lines + infrastructure |
| Runs in Colab for 30 minutes | Runs 24/7 on servers, handles millions of queries |

### But the Core Pattern Is Identical

```
1. Receive a research question
2. THINK about what information is needed
3. ACT by calling the right tool
4. OBSERVE the results and evaluate quality
5. REPEAT until you have enough high-quality data
6. SYNTHESIZE into a structured answer
```

You built the same architecture. Production just adds:
- **Real APIs** instead of simulated data
- **Persistent memory** across sessions
- **Multiple agents** collaborating (researcher + writer + editor)
- **MCP (Model Context Protocol)** for standardized tool connections
- **Error handling** and retries for reliability
- **Security** and sandboxing for safety

### 💡 The Business Insight

The **most valuable skill** isn't coding the ReAct loop (that's ~80 lines). It's:

1. **Choosing what to research** — Which business questions benefit from agent-powered research?
2. **Designing the right tools** — What data sources matter? What's the evaluation criteria?
3. **Evaluating the output** — Is the agent's research actually *better* than a human analyst?
4. **Knowing when to stop** — When does more automation hurt rather than help?

These are business decisions, not engineering decisions. That's why **you** — as business students — are perfectly positioned to design the next generation of AI agents.

### Real-World Examples

- **NanoClaw** (500 lines of code) uses this exact ReAct pattern with MCP tools
- **Claude Code** reasons through multi-step coding tasks with the same think-act-observe loop
- **Perplexity** searches, evaluates, and synthesizes — exactly what your agent does
- **Devin** (AI software engineer) uses ReAct to plan, code, test, and debug

You now understand the architecture behind all of them.

---

## 🚀 What's Next?

- **Homework**: Add a 4th tool to this notebook. Test it with 3 different research questions. Document the reasoning traces.
- **Session 9**: Ethics & Responsible AI — agents are powerful, but power needs guardrails

---

**ISOM 260: AI for Business** | Suffolk University | Session 8

🌐 [isom-260.vercel.app](https://isom-260.vercel.app)